# PromptGFM-Bio — ARC Labs Workstation Notebook
**Gene-Phenotype Prediction for Rare Diseases**

| Spec | Value |
|---|---|
| CPU | Intel i9-14900K |
| RAM | 128 GB |
| GPU | NVIDIA GeForce RTX 4090 (24 GB VRAM) |
| CUDA | 13.0 · Driver 580.65.06 |
| Disk | 512 GB |

### How this notebook works
- **Single project root**: every cell uses `PROJECT_ROOT` — no path confusion.
- **VRAM-aware**: auto-detects free GPU memory and scales batch size accordingly.
- **Resumable**: data persists on disk between Jupyter sessions (5-day window).
- **Secrets**: GitHub token + W&B key loaded from `.env` in the project root.

> ⚠️ **5-day data retention** — back up to GitHub/HuggingFace before your window expires.

## 0. 🔒 Master Path Setup (RUN THIS FIRST — ALWAYS)

This cell sets `PROJECT_ROOT` and `os.chdir()` so **every cell below**
uses the same directory. Nothing else in this notebook defines paths independently.

In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS ONE LINE if your project is in a different location         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, sys, subprocess
from pathlib import Path

GITHUB_URL   = "https://github.com/pes1ug23am910/PromptGFM-Bio.git"
PROJECT_ROOT = Path("/home/mluser/projects_yash/new_project/PromptGFM-Bio").resolve()

# ── Ensure directory exists ───────────────────────────────────────────────
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# ═══════════════════════════════════════════════════════════════════════════
# AUTO-CLONE: if scripts/ and configs/ are missing, pull the repo code
# Safe for non-empty directories — won't clobber .env, data/, hf_cache/
# ═══════════════════════════════════════════════════════════════════════════
if not (PROJECT_ROOT / "scripts").is_dir() or not (PROJECT_ROOT / "configs").is_dir():
    print("⚠️  scripts/ and/or configs/ not found — cloning repo code...")
    print()

    # Load .env early to get GITHUB_TOKEN for private repo
    _env = PROJECT_ROOT / ".env"
    if _env.is_file():
        for _line in _env.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _, _v = _line.partition("=")
                os.environ[_k.strip()] = _v.strip()

    _token = os.environ.get("GITHUB_TOKEN", "")
    _auth_url = GITHUB_URL.replace("https://", f"https://{_token}@") if _token else GITHUB_URL

    if (PROJECT_ROOT / ".git").is_dir():
        # Already a git repo but missing files — just pull
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=True)
    else:
        # Not a git repo — init and pull (safe for non-empty dirs)
        subprocess.run(["git", "init"], cwd=str(PROJECT_ROOT), check=True,
                       capture_output=True)
        subprocess.run(["git", "remote", "add", "origin", _auth_url],
                       cwd=str(PROJECT_ROOT), capture_output=True)

        result = subprocess.run(
            ["git", "fetch", "origin"],
            cwd=str(PROJECT_ROOT), capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"   ❌ git fetch failed: {result.stderr.strip()}")
            if not _token:
                print("   Repo is private — add GITHUB_TOKEN to .env")
            raise RuntimeError("Cannot fetch repo code")

        # Detect default branch
        _branch_result = subprocess.run(
            ["git", "remote", "show", "origin"],
            cwd=str(PROJECT_ROOT), capture_output=True, text=True
        )
        _default_branch = "main"
        for _bl in _branch_result.stdout.splitlines():
            if "HEAD branch" in _bl:
                _default_branch = _bl.split(":")[-1].strip()
                break

        subprocess.run(
            ["git", "checkout", "-f", f"origin/{_default_branch}", "-b", _default_branch],
            cwd=str(PROJECT_ROOT), capture_output=True
        )

    if (PROJECT_ROOT / "scripts").is_dir() and (PROJECT_ROOT / "configs").is_dir():
        print("   ✅ Repo code cloned successfully")
    else:
        print("   ❌ Clone finished but scripts/configs still missing!")
        raise RuntimeError("Repo doesn't contain expected scripts/ and configs/")
else:
    print("✅ Repo code already present (scripts/ and configs/ found)")

# ── Derived path variables ────────────────────────────────────────────────
SCRIPTS_DIR  = PROJECT_ROOT / "scripts"
CONFIGS_DIR  = PROJECT_ROOT / "configs"
DATA_DIR     = PROJECT_ROOT / "data"
CKPT_DIR     = PROJECT_ROOT / "checkpoints" / "promptgfm_film"
LOGS_DIR     = PROJECT_ROOT / "logs"
ENV_FILE     = PROJECT_ROOT / ".env"

# ── HF cache ─────────────────────────────────────────────────────────────
HF_CACHE_DIR = str(PROJECT_ROOT / "hf_cache")
os.environ["HF_HOME"]               = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"]     = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE_DIR

# ── Resume flags ──────────────────────────────────────────────────────────
RESUME_HF_CACHE     = False
RESUME_DATA         = False
RESUME_CHECKPOINTS  = False

# ── Status report ─────────────────────────────────────────────────────────
print()
print(f"PROJECT_ROOT   = {PROJECT_ROOT}")
print(f"CWD            = {Path.cwd()}")
print(f"  scripts/     : {'✅' if SCRIPTS_DIR.is_dir() else '❌'}")
print(f"  configs/     : {'✅' if CONFIGS_DIR.is_dir() else '❌'}")
print(f"  data/        : {'✅' if DATA_DIR.is_dir() else '⬜ will create'}")
print(f"  .env         : {'✅' if ENV_FILE.is_file() else '❌ MISSING'}")
print(f"  .git/        : {'✅' if (PROJECT_ROOT / '.git').is_dir() else '❌'}")
print(f"  hf_cache/    : {'✅' if Path(HF_CACHE_DIR).is_dir() else '⬜ will create'}")
print(f"  RESUME_HF_CACHE    = {RESUME_HF_CACHE}")
print(f"  RESUME_DATA        = {RESUME_DATA}")
print(f"  RESUME_CHECKPOINTS = {RESUME_CHECKPOINTS}")

✅ Repo code already present (scripts/ and configs/ found)

PROJECT_ROOT   = /home/mluser/projects_yash/new_project/PromptGFM-Bio
CWD            = /home/mluser/projects_yash/new_project/PromptGFM-Bio
  scripts/     : ✅
  configs/     : ✅
  data/        : ✅
  .env         : ✅
  .git/        : ❌
  hf_cache/    : ✅
  RESUME_HF_CACHE    = False
  RESUME_DATA        = False
  RESUME_CHECKPOINTS = False


## 1. Load Secrets from `.env`

Your `.env` file should be at:
```
/home/mluser/projects_yash/new_project/PromptGFM-Bio/.env
```
Contents:
```
GITHUB_TOKEN=ghp_xxxxxxxxxxxxxxxxxxxx
WANDB_API_KEY=xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

In [2]:
import os

print(f"Loading: {ENV_FILE}")

if ENV_FILE.is_file():
    for line in ENV_FILE.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, val = line.partition("=")
            os.environ[key.strip()] = val.strip()
    print("✅ .env loaded")
else:
    print(f"❌ .env not found at {ENV_FILE}")
    print(f'   Create it:  echo -e "GITHUB_TOKEN=ghp_xxx\nWANDB_API_KEY=xxx" > {ENV_FILE}')

print(f"  GITHUB_TOKEN  : {'✅ set' if os.environ.get('GITHUB_TOKEN') else '❌ missing'}")
print(f"  WANDB_API_KEY : {'✅ set' if os.environ.get('WANDB_API_KEY') else '❌ missing'}")

Loading: /home/mluser/projects_yash/new_project/PromptGFM-Bio/.env
✅ .env loaded
  GITHUB_TOKEN  : ✅ set
  WANDB_API_KEY : ✅ set


## 2. GPU Diagnostics & VRAM Budget

In [3]:
import subprocess, os

subprocess.run(["nvidia-smi"])
print()

try:
    import torch
    if torch.cuda.is_available():
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total,memory.free",
             "--format=csv,nounits,noheader"],
            capture_output=True, text=True
        )
        smi_used, smi_total, smi_free = [int(x.strip()) for x in result.stdout.strip().split(",")]
        usable_mb = smi_free - 2048

        print(f"GPU           : {torch.cuda.get_device_name(0)}")
        print(f"Total VRAM    : {smi_total:,} MiB")
        print(f"Used (idle)   : {smi_used:,} MiB")
        print(f"Free VRAM     : {smi_free:,} MiB")
        print(f"Usable        : {usable_mb:,} MiB (after 2 GB safety margin)")
        print(f"PyTorch       : {torch.__version__}")
        print(f"CUDA          : {torch.version.cuda}")

        os.environ["_VRAM_FREE_MB"]   = str(smi_free)
        os.environ["_VRAM_USABLE_MB"] = str(usable_mb)
    else:
        print("⚠️  No GPU detected")
except ImportError:
    print("⚠️  PyTorch not installed yet — run Step 3 first")

Thu Apr  9 13:06:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:01:00.0 Off |                  Off |
| 34%   40C    P8             18W /  450W |     108MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Install PyTorch & PyTorch Geometric

Installs into your existing venv. CUDA 12.4 wheels are compatible with your CUDA 13.0 driver.

In [4]:
import subprocess, sys

# ── PyTorch ───────────────────────────────────────────────────────────────
try:
    import torch
    print(f"PyTorch {torch.__version__} already installed (CUDA {torch.version.cuda})")
except ImportError:
    print("Installing PyTorch...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "torch", "torchvision", "torchaudio",
        "--index-url", "https://download.pytorch.org/whl/cu124"
    ], check=True)
    import torch
    print(f"✅ PyTorch {torch.__version__} installed")

# ── PyG ───────────────────────────────────────────────────────────────────
TORCH_VER = torch.__version__.split("+")[0]
CUDA_VER  = torch.version.cuda.replace(".", "")
WHEEL_URL = f"https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html"
print(f"PyG wheel source: {WHEEL_URL}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "-f", WHEEL_URL,
     "torch-scatter", "torch-sparse", "torch-cluster",
     "torch-spline-conv", "torch-geometric"],
    check=True
)
print("✅ PyTorch Geometric installed")

PyTorch 2.6.0+cu124 already installed (CUDA 12.4)
PyG wheel source: https://data.pyg.org/whl/torch-2.6.0+cu124.html
✅ PyTorch Geometric installed


In [5]:
'''
#New in TE v6
# ══════════════════════════════════════════════════════════════════════════
# 3.5  PyG Extension Compatibility Check
# Detects ABI mismatches so we know whether scatter/sparse ops use
# fast CUDA kernels or fall back to pure-PyTorch paths.
# ══════════════════════════════════════════════════════════════════════════
import warnings, importlib, sys, subprocess
import torch

_EXT = {
    "pyg-lib":            "pyg_lib",
    "torch-scatter":      "torch_scatter",
    "torch-sparse":       "torch_sparse",
    "torch-cluster":      "torch_cluster",
    "torch-spline-conv":  "torch_spline_conv",
}

_broken, _missing = [], []
for pkg, mod in _EXT.items():
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", UserWarning)
            importlib.import_module(mod)
        print(f"  ✅ {pkg:<24} C++ kernels active")
    except UserWarning:
        _broken.append(pkg)
        print(f"  ⚠️  {pkg:<24} ABI mismatch — pure-PyTorch fallback")
    except ImportError:
        _missing.append(pkg)
        print(f"  ❌ {pkg:<24} not installed")

_bad = _broken + _missing
if _bad:
    print(f"\n{len(_bad)} extension(s) degraded.  Impact on this project:")
    print("  • torch-scatter / torch-sparse: scatter_add falls back to native PyTorch")
    print("    → GraphSAGE trains correctly; scatter ops ~10-20% slower")
    print("  • pyg-lib / torch-spline-conv:  not used by GraphSAGE — no impact")
    print("\nAttempting force-reinstall from PyG CDN ...")
    _whl = (f"https://data.pyg.org/whl/"
            f"torch-{torch.__version__.split('+')[0]}"
            f"+cu{torch.version.cuda.replace('.','')}.html")
    _r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall",
         "-f", _whl,
         "torch-scatter", "torch-sparse", "torch-cluster",
         "torch-spline-conv", "torch-geometric"],
        capture_output=True, text=True,
    )
    if _r.returncode == 0:
        print("✅ Reinstall succeeded — restart kernel then re-run this cell to verify")
    else:
        print("⚠️  Pre-compiled wheels not yet available for this PyTorch/CUDA version.")
        print("   Training will proceed with pure-PyTorch fallbacks (fully correct).")
        print("   To silence the flood of UserWarnings during training, the patch cell")
        print("   below will suppress them at import time.")
else:
    print("\n✅ All PyG extensions functional")
'''

'\n#New in TE v6\n# ══════════════════════════════════════════════════════════════════════════\n# 3.5  PyG Extension Compatibility Check\n# Detects ABI mismatches so we know whether scatter/sparse ops use\n# fast CUDA kernels or fall back to pure-PyTorch paths.\n# ══════════════════════════════════════════════════════════════════════════\nimport warnings, importlib, sys, subprocess\nimport torch\n\n_EXT = {\n    "pyg-lib":            "pyg_lib",\n    "torch-scatter":      "torch_scatter",\n    "torch-sparse":       "torch_sparse",\n    "torch-cluster":      "torch_cluster",\n    "torch-spline-conv":  "torch_spline_conv",\n}\n\n_broken, _missing = [], []\nfor pkg, mod in _EXT.items():\n    try:\n        with warnings.catch_warnings():\n            warnings.simplefilter("error", UserWarning)\n            importlib.import_module(mod)\n        print(f"  ✅ {pkg:<24} C++ kernels active")\n    except UserWarning:\n        _broken.append(pkg)\n        print(f"  ⚠️  {pkg:<24} ABI mismatch — pure

In [6]:
#New in TE v6
# ══════════════════════════════════════════════════════════════════════════
# 3.5  PyG Extension Compatibility Check
# Detects ABI mismatches so we know whether scatter/sparse ops use
# fast CUDA kernels or fall back to pure-PyTorch paths.
# ══════════════════════════════════════════════════════════════════════════
import warnings, importlib, sys, subprocess
import torch

_EXT = {
    "pyg-lib":           "pyg_lib",
    "torch-scatter":     "torch_scatter",
    "torch-sparse":      "torch_sparse",
    "torch-cluster":     "torch_cluster",
    "torch-spline-conv": "torch_spline_conv",
}

_broken, _missing = [], []

for pkg, mod in _EXT.items():
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", UserWarning)
            importlib.import_module(mod)
        print(f"  ✅ {pkg:<24} C++ kernels active")
    except (UserWarning, OSError):
        _broken.append(pkg)
        print(f"  ⚠️  {pkg:<24} ABI mismatch — pure-PyTorch fallback")
    except ImportError:
        _missing.append(pkg)
        print(f"  ❌ {pkg:<24} not installed")

_bad = _broken + _missing

if _bad:
    print(f"\n{len(_bad)} extension(s) degraded. Impact on this project:")
    print("  • torch-scatter / torch-sparse: scatter_add falls back to native PyTorch")
    print("    → GraphSAGE trains correctly; scatter ops ~10-20% slower")
    print("  • pyg-lib / torch-spline-conv: not used by GraphSAGE — no impact")

    print("\nAttempting force-reinstall from PyG CDN ...")

    torch_version = torch.__version__.split("+")[0]
    cuda_version = torch.version.cuda

    if cuda_version is None:
        wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+cpu.html"
    else:
        wheel_url = (
            f"https://data.pyg.org/whl/"
            f"torch-{torch_version}+cu{cuda_version.replace('.', '')}.html"
        )

    _r = subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--quiet", "--force-reinstall", "--no-cache-dir",
            "-f", wheel_url,
            "pyg_lib", "torch_scatter", "torch_sparse",
            "torch_cluster", "torch_spline_conv", "torch_geometric",
        ],
        capture_output=True,
        text=True,
    )

    if _r.returncode == 0:
        print("✅ Reinstall succeeded — restart kernel then re-run this cell to verify")
    else:
        print("⚠️  Pre-compiled wheels not yet available for this PyTorch/CUDA version.")
        print("   Training will proceed with pure-PyTorch fallbacks (fully correct).")
        print("   To silence the flood of UserWarnings during training, the patch cell")
        print("   below will suppress them at import time.")
else:
    print("\n✅ All PyG extensions functional")

  ✅ pyg-lib                  C++ kernels active
  ✅ torch-scatter            C++ kernels active
  ✅ torch-sparse             C++ kernels active
  ✅ torch-cluster            C++ kernels active
  ✅ torch-spline-conv        C++ kernels active

✅ All PyG extensions functional


## 4. Install Extra Dependencies

In [7]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "--upgrade", "setuptools", "wheel", "pip"], check=True)

extra = [
    "transformers>=4.40.0",
    "sentence-transformers>=2.7.0",
    "biopython>=1.84",
    "networkx>=3.2",
    "wandb>=0.17.0",
    "python-dotenv>=1.0.0",
    "huggingface_hub",
    "pyyaml",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + extra, check=True)
print("✅ Extra packages installed")

✅ Extra packages installed


## 5. Git Pull Latest Code

Your repo is already cloned at `PROJECT_ROOT`. This just pulls the latest changes.

In [8]:
import subprocess, os

os.environ["GIT_TERMINAL_PROMPT"] = "0"

if (PROJECT_ROOT / ".git").is_dir():
    result = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("✅ Latest code pulled")
        if result.stdout.strip():
            print(f"   {result.stdout.strip()}")
    else:
        print("⚠️  git pull failed — continuing with existing code")
        print(f"   {result.stderr.strip()}")
else:
    print("⚠️  Not a git repo — skipping pull (code was set up in Cell 0)")

⚠️  Not a git repo — skipping pull (code was set up in Cell 0)


In [9]:
'''
import os, sys, subprocess
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ EDIT THIS LINE to match your home directory / preferred location ║
# ╚══════════════════════════════════════════════════════════════════════════╝
GITHUB_URL = 'https://github.com/pes1ug23am910/PROMPTGMF-Bio-Kaggle.git'
PROJECT_DIR = Path('/home/mluser/projects_yash/new_project/PromptGFM-Bio').resolve() # ← change this

# ── Ensure parent directory exists ───────────────────────────────────────
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# ── Clone or pull ─────────────────────────────────────────────────────────
if not (PROJECT_DIR / '.git').is_dir():
 subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, str(PROJECT_DIR)], check=True)
 print(f' Cloned to {PROJECT_DIR}')
else:
 subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull'], check=True)
 print(f' Pulled latest changes')

# ── Set working directory & Python path ──────────────────────────────────
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print(f'Working directory: {os.getcwd()}')

'''

"\nimport os, sys, subprocess\nfrom pathlib import Path\n\n# ╔══════════════════════════════════════════════════════════════════════════╗\n# ║ EDIT THIS LINE to match your home directory / preferred location ║\n# ╚══════════════════════════════════════════════════════════════════════════╝\nGITHUB_URL = 'https://github.com/pes1ug23am910/PROMPTGMF-Bio-Kaggle.git'\nPROJECT_DIR = Path('/home/mluser/projects_yash/new_project/PromptGFM-Bio').resolve() # ← change this\n\n# ── Ensure parent directory exists ───────────────────────────────────────\nPROJECT_DIR.mkdir(parents=True, exist_ok=True)\n\n# ── Clone or pull ─────────────────────────────────────────────────────────\nif not (PROJECT_DIR / '.git').is_dir():\n subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, str(PROJECT_DIR)], check=True)\n print(f' Cloned to {PROJECT_DIR}')\nelse:\n subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull'], check=True)\n print(f' Pulled latest changes')\n\n# ── Set working directory & Python path 

## 6. Create Directory Structure

In [10]:
for d in [DATA_DIR / "raw", DATA_DIR / "processed", DATA_DIR / "splits", CKPT_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("✅ Directories created")

✅ Directories created


## 7. Pre-download BioBERT Model
Downloads ~440 MB of PubMedBERT weights. Only needed once — cache persists on disk.

In [11]:
from huggingface_hub import snapshot_download
from pathlib import Path
import warnings, os

MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

hub_dir = Path(HF_CACHE_DIR) / "hub"
model_cache_name = "models--" + MODEL_NAME.replace("/", "--")
model_snapshots = hub_dir / model_cache_name / "snapshots"

if RESUME_HF_CACHE and model_snapshots.exists() and any(model_snapshots.iterdir()):
    print("⏭️  BioBERT already cached — skipping download")
else:
    print(f"Downloading {MODEL_NAME}")
    print(f"  Cache dir : {HF_CACHE_DIR}")
    print(f"  Size      : ~440 MB")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        snapshot_download(
            repo_id=MODEL_NAME,
            cache_dir=HF_CACHE_DIR,
            ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*", "rust_model*", "*.ot"],
        )
    print("\n✅ BioBERT fully downloaded and cached")

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"
print("✅ Offline mode enabled")

  Cache dir : /home/mluser/projects_yash/new_project/PromptGFM-Bio/hf_cache
  Size      : ~440 MB


/home/mluser/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 7 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 33100.48it/s]


✅ BioBERT fully downloaded and cached
✅ Offline mode enabled


## 8. Download Biomedical Datasets
Skipped if `RESUME_DATA=True` and graph exists. Total download: ~1.5 GB.

In [12]:
import subprocess, sys

graph_exists = (DATA_DIR / "processed" / "biomedical_graph.pt").exists()

if RESUME_DATA and graph_exists:
    print("⏭️  Download skipped — data already on disk")
else:
    print("Downloading datasets...")
    script = str(SCRIPTS_DIR / "download_data.py")
    print(f"  Running: {script}")
    result = subprocess.run(
        [sys.executable, script, "--dataset", "all"],
        cwd=str(PROJECT_ROOT),
    )
    print("Download exit code:", result.returncode)
    if result.returncode != 0:
        print("⚠️  Download may have failed — check output above")

  Running: /home/mluser/projects_yash/new_project/PromptGFM-Bio/scripts/download_data.py

PromptGFM-Bio Data Download

Dataset: all
Force re-download: False
Skip failing: True


✓ Successfully downloaded 4/4 datasets

✓ DOWNLOAD COMPLETE

Next steps:
  1. Run preprocessing: python scripts/preprocess_all.py
  2. Check downloaded files in: data/raw/

Download exit code: 0


INFO:src.data.download:======================================================================
INFO:src.data.download:Starting download of all biomedical datasets...
INFO:src.data.download:This may take 30-60 minutes depending on your connection
INFO:src.data.download:Total size: ~1.5 GB
INFO:src.data.download:======================================================================
INFO:src.data.download:
[1/4] BioGRID Protein-Protein Interactions
INFO:src.data.download:BioGRID file already exists at /home/mluser/projects_yash/new_project/PromptGFM-Bio/data/raw/biogrid/BIOGRID-ALL-4.4.224.tab3.zip
INFO:src.data.download:Use force=True to re-download
INFO:src.data.download:
[2/4] STRING Protein Network
INFO:src.data.download:STRING file already exists at /home/mluser/projects_yash/new_project/PromptGFM-Bio/data/raw/string/9606.protein.links.v12.0.txt.gz
INFO:src.data.download:Use force=True to re-download
INFO:src.data.download:
[3/4] DisGeNET Gene-Disease Associations
INFO:src.data.downlo

## 9. Preprocess Data (Build Knowledge Graph)
Skipped if `RESUME_DATA=True` and graph exists.

In [13]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "pandas"], check=True)

CompletedProcess(args=['/home/mluser/.venv/bin/python3', '-m', 'pip', 'install', 'pandas'], returncode=0)

In [14]:
import subprocess, sys
from pathlib import Path

graph_path = DATA_DIR / "processed" / "biomedical_graph.pt"
script = str(SCRIPTS_DIR / "preprocess_all.py")

if RESUME_DATA and graph_path.exists():
    print(f"⏭️  Preprocessing skipped — graph ready ({graph_path.stat().st_size/1e6:.0f} MB)")
else:
    cmd = [sys.executable, script]

    # If a previous graph exists, force rebuild so STRING path/mapping fixes are applied.
    if graph_path.exists():
        cmd.append("--force")
        print("Existing graph found — forcing rebuild to refresh PPI edges")

    print("Building knowledge graph...")
    print(f"  Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    print("Preprocessing exit code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Preprocessing failed — check logs above")

if not graph_path.exists():
    raise RuntimeError("Graph file not created — check logs above")

print(f"✅ Graph ready ({graph_path.stat().st_size/1e6:.0f} MB)")

# Validate that gene-gene edges exist for GNN message passing.
import torch

def _load_graph_cpu(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def _count_ppi_edges(graph_obj):
    edge_types = [
        ("gene", "interacts", "gene"),
        ("gene", "protein_interaction", "gene"),
        ("gene", "ppi", "gene"),
    ]
    for edge_type in edge_types:
        if edge_type in graph_obj.edge_types:
            return int(graph_obj[edge_type].edge_index.shape[1]), edge_type
    return 0, None

graph = _load_graph_cpu(graph_path)
ppi_edge_count, matched_edge_type = _count_ppi_edges(graph)

if ppi_edge_count > 0:
    print(f"✅ Gene-gene edges found: {ppi_edge_count:,} ({matched_edge_type})")
else:
    print("⚠️  No gene-gene edges found after preprocessing.")
    patch_script = PROJECT_ROOT / "add_string_ppi_edges.py"

    if patch_script.exists():
        print(f"  Running fallback patch: {patch_script}")
        patch_result = subprocess.run(
            [sys.executable, str(patch_script)],
            cwd=str(PROJECT_ROOT),
        )
        if patch_result.returncode != 0:
            raise RuntimeError("Fallback PPI patch failed — check logs above")

        graph = _load_graph_cpu(graph_path)
        repaired_count, repaired_edge_type = _count_ppi_edges(graph)
        if repaired_count > 0:
            print(f"✅ Repaired gene-gene edges: {repaired_count:,} ({repaired_edge_type})")
        else:
            raise RuntimeError("Graph still has no gene-gene edges after fallback patch")
    else:
        raise RuntimeError(f"Missing fallback script: {patch_script}")

Existing graph found — forcing rebuild to refresh PPI edges
Building knowledge graph...
  Running: /home/mluser/.venv/bin/python3 /home/mluser/projects_yash/new_project/PromptGFM-Bio/scripts/preprocess_all.py --force


INFO:src.data.preprocess:======================================================================
INFO:src.data.preprocess:Starting enhanced preprocessing pipeline...
INFO:src.data.preprocess:======================================================================
INFO:src.data.preprocess:Options:
INFO:src.data.preprocess:  HPO Bridge: True
INFO:src.data.preprocess:  Orphadata: True
INFO:src.data.preprocess:  UniProt: False
INFO:src.data.preprocess:  Pathways: False
INFO:src.data.preprocess:======================================================================
INFO:src.data.preprocess:
[Step 1] Parsing PPI networks...
INFO:src.data.preprocess:Using STRING links file: /home/mluser/projects_yash/new_project/PromptGFM-Bio/data/raw/string/9606.protein.links.v12.0.txt
INFO:src.data.preprocess:Using STRING protein info file: /home/mluser/projects_yash/new_project/PromptGFM-Bio/data/raw/string/9606.protein.info.v12.0.txt
INFO:src.data.preprocess:Parsing BioGRID from /home/mluser/projects_yash/new


PromptGFM-Bio Enhanced Preprocessing Pipeline

Configuration:
  Force reprocess: True
  HPO Bridge: True
  Orphadata: True
  UniProt: False
  Pathways: False


✓ PREPROCESSING COMPLETE

Next steps:
  1. Create dataset splits: python -m src.data.dataset
  2. Check graph file: data/processed/biomedical_graph.pt
  3. View statistics: data/processed/biomedical_graph_stats.txt

Preprocessing exit code: 0
✅ Graph ready (351 MB)
✅ Gene-gene edges found: 1,854,012 (('gene', 'interacts', 'gene'))


In [15]:
'''
#use if needed/ previous cell fails
# ─────────────────────────────────────────────────────────────
# 4. Install & Verify All Dependencies (ROBUST VERSION)
# ─────────────────────────────────────────────────────────────

import subprocess, sys, importlib

def pip_install(packages):
    print(f"Installing: {packages}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
        check=True
    )

def ensure_import(pkg_name, pip_name=None):
    """Ensure a package is installed and importable"""
    pip_name = pip_name or pkg_name
    try:
        importlib.import_module(pkg_name)
        print(f"✅ {pkg_name} already available")
    except ImportError:
        print(f"⚠️  {pkg_name} missing → installing...")
        pip_install([pip_name])
        try:
            importlib.import_module(pkg_name)
            print(f"✅ {pkg_name} installed successfully")
        except ImportError:
            raise RuntimeError(f"❌ Failed to import {pkg_name} even after install")

# ── Upgrade core tooling ──────────────────────────────────────
pip_install(["--upgrade", "pip", "setuptools", "wheel"])

# ── Critical deps (MUST exist for scripts to run) ─────────────
critical_deps = {
    "pandas": "pandas>=2.2.0",        # 🔴 REQUIRED (your failure)
    "numpy": "numpy",
    "yaml": "pyyaml",
}

# ── Project deps ─────────────────────────────────────────────
project_deps = {
    "transformers": "transformers>=4.40.0",
    "sentence_transformers": "sentence-transformers>=2.7.0",
    "Bio": "biopython>=1.84",
    "networkx": "networkx>=3.2",
    "wandb": "wandb>=0.17.0",
    "dotenv": "python-dotenv>=1.0.0",
    "huggingface_hub": "huggingface_hub",
}

# ── Ensure everything is importable ───────────────────────────
print("\n🔍 Checking critical dependencies...")
for module, pip_pkg in critical_deps.items():
    ensure_import(module, pip_pkg)

print("\n🔍 Checking project dependencies...")
for module, pip_pkg in project_deps.items():
    ensure_import(module, pip_pkg)

print("\n✅ All dependencies verified and ready")
'''

'\n#use if needed/ previous cell fails\n# ─────────────────────────────────────────────────────────────\n# 4. Install & Verify All Dependencies (ROBUST VERSION)\n# ─────────────────────────────────────────────────────────────\n\nimport subprocess, sys, importlib\n\ndef pip_install(packages):\n    print(f"Installing: {packages}")\n    subprocess.run(\n        [sys.executable, "-m", "pip", "install", "--quiet"] + packages,\n        check=True\n    )\n\ndef ensure_import(pkg_name, pip_name=None):\n    """Ensure a package is installed and importable"""\n    pip_name = pip_name or pkg_name\n    try:\n        importlib.import_module(pkg_name)\n        print(f"✅ {pkg_name} already available")\n    except ImportError:\n        print(f"⚠️  {pkg_name} missing → installing...")\n        pip_install([pip_name])\n        try:\n            importlib.import_module(pkg_name)\n            print(f"✅ {pkg_name} installed successfully")\n        except ImportError:\n            raise RuntimeError(f"❌ Fail

## 10. W&B Login

In [16]:
import os

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print("✅ W&B logged in")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("W&B disabled — add WANDB_API_KEY to .env to enable")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/mluser/.netrc
wandb: Currently logged in as: pes1ug23am910 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B logged in


## 11. 🧠 Generate VRAM-Aware Config

Reads `configs/kaggle_config.yaml` and patches it based on free VRAM **right now**.

| Free VRAM | batch_size | grad_accum | effective_batch | workers |
|---|---|---|---|---|
| ≥ 20 GB | 768 | 1 | 768 | 8 |
| 16–20 GB | 512 | 1 | 512 | 6 |
| 12–16 GB | 384 | 1 | 384 | 4 |
| 8–12 GB | 256 | 1 | 256 | 4 |
| 5–8 GB | 128 | 2 | 256 | 2 |
| < 5 GB | 64 | 4 | 256 | 2 |

In [17]:
'''
#Modified to fix workers in TE v6
import subprocess, os, yaml
import torch

# ── 1. Probe free VRAM ───────────────────────────────────────────────────
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total,memory.free",
     "--format=csv,nounits,noheader"],
    capture_output=True, text=True
)
smi_used, smi_total, smi_free = [int(x.strip()) for x in result.stdout.strip().split(",")]
usable_mb = smi_free - 2048

print(f"GPU VRAM: {smi_total:,} MiB total · {smi_used:,} MiB used · {smi_free:,} MiB free")
print(f"Usable for training: {usable_mb:,} MiB (after 2 GB safety margin)")
print()

# ── 2. Pick batch_size + gradient accumulation ───────────────────────────
if usable_mb >= 20000:
    batch_size, grad_accum, num_workers = 768, 1, 8
elif usable_mb >= 16000:
    batch_size, grad_accum, num_workers = 512, 1, 6
elif usable_mb >= 12000:
    batch_size, grad_accum, num_workers = 384, 1, 4
elif usable_mb >= 8000:
    batch_size, grad_accum, num_workers = 256, 1, 4
elif usable_mb >= 5000:
    batch_size, grad_accum, num_workers = 128, 2, 2
else:
    batch_size, grad_accum, num_workers = 64, 4, 2

effective_batch = batch_size * grad_accum

print(f"╔══════════════════════════════════════════════╗")
print(f"║  batch_size       = {batch_size:<6}                   ║")
print(f"║  grad_accum       = {grad_accum:<6}                   ║")
print(f"║  effective_batch   = {effective_batch:<6}                  ║")
print(f"║  num_workers      = {num_workers:<6}                   ║")
print(f"╚══════════════════════════════════════════════╝")
print()

# ── 3. Read base config and patch ────────────────────────────────────────
base_cfg_path = CONFIGS_DIR / "kaggle_config.yaml"
ws_cfg_path   = CONFIGS_DIR / "workstation_config.yaml"

if base_cfg_path.exists():
    with open(base_cfg_path) as f:
        cfg = yaml.safe_load(f)

    def patch_dict(d, patches):
        if not isinstance(d, dict):
            return
        for k, v in patches.items():
            if k in d:
                print(f"  patching {k}: {d[k]} → {v}")
                d[k] = v
        for child in d.values():
            if isinstance(child, dict):
                patch_dict(child, patches)

    patches = {
        "batch_size":                   batch_size,
        "num_workers":                  num_workers,
        "gradient_accumulation_steps":  grad_accum,
        "pin_memory":                   True,
        "persistent_workers":           num_workers > 0,
    }

    print("Patching config:")
    patch_dict(cfg, patches)

    if "gradient_accumulation_steps" not in str(cfg):
        target = cfg.get("training", cfg)
        target["gradient_accumulation_steps"] = grad_accum
        print(f"  added gradient_accumulation_steps: {grad_accum} (new key)")

    with open(ws_cfg_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

    print(f"\n✅ Wrote {ws_cfg_path}")
else:
    print(f"⚠️  {base_cfg_path} not found")

# ── 4. RTX 4090 performance flags ────────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True
print("✅ TF32 matmul ✓ · TF32 cuDNN ✓ · cuDNN benchmark ✓")
'''

'\n#Modified to fix workers in TE v6\nimport subprocess, os, yaml\nimport torch\n\n# ── 1. Probe free VRAM ───────────────────────────────────────────────────\nresult = subprocess.run(\n    ["nvidia-smi", "--query-gpu=memory.used,memory.total,memory.free",\n     "--format=csv,nounits,noheader"],\n    capture_output=True, text=True\n)\nsmi_used, smi_total, smi_free = [int(x.strip()) for x in result.stdout.strip().split(",")]\nusable_mb = smi_free - 2048\n\nprint(f"GPU VRAM: {smi_total:,} MiB total · {smi_used:,} MiB used · {smi_free:,} MiB free")\nprint(f"Usable for training: {usable_mb:,} MiB (after 2 GB safety margin)")\nprint()\n\n# ── 2. Pick batch_size + gradient accumulation ───────────────────────────\nif usable_mb >= 20000:\n    batch_size, grad_accum, num_workers = 768, 1, 8\nelif usable_mb >= 16000:\n    batch_size, grad_accum, num_workers = 512, 1, 6\nelif usable_mb >= 12000:\n    batch_size, grad_accum, num_workers = 384, 1, 4\nelif usable_mb >= 8000:\n    batch_size, grad_a

In [18]:
#New in TE v6
import subprocess, os, yaml
import torch

# ── 1. Probe free VRAM ───────────────────────────────────────────────────
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total,memory.free",
     "--format=csv,nounits,noheader"],
    capture_output=True, text=True
)
smi_used, smi_total, smi_free = [int(x.strip()) for x in result.stdout.strip().split(",")]
usable_mb = smi_free - 2048

print(f"GPU VRAM: {smi_total:,} MiB total · {smi_used:,} MiB used · {smi_free:,} MiB free")
print(f"Usable for training: {usable_mb:,} MiB (after 2 GB safety margin)")
print()

# ── 2. Probe /dev/shm free space ─────────────────────────────────────────
# Each DataLoader worker serialises ~50 MB of tensors (node_features +
# edge_index) per batch prefetch via /dev/shm when multiprocessing_context
# is the default 'fork-server' / 'spawn'.  The patch cell below switches to
# 'fork', which inherits parent memory and bypasses /dev/shm entirely.
# We still record it here as a diagnostic.
_shm_r = subprocess.run(
    "df -m /dev/shm | awk 'NR==2 {print $2, $4}'",
    shell=True, capture_output=True, text=True
)
try:
    shm_total_mb, shm_free_mb = [int(x) for x in _shm_r.stdout.strip().split()]
except ValueError:
    shm_total_mb, shm_free_mb = 64, 64   # conservative fallback

WORKER_SHM_MB = 50   # approx shm consumption per worker (pre-fork-patch)
shm_safe_workers = max(0, shm_free_mb // WORKER_SHM_MB - 1)

print(f"/dev/shm  total={shm_total_mb} MB  free={shm_free_mb} MB")
if shm_free_mb < 256:
    print(f"  ⚠️  /dev/shm is tight ({shm_free_mb} MB free).  Without the fork-context")
    print(f"     patch, multi-worker DataLoading WILL crash (Bus Error / errno 28).")
    print(f"     Run the 'Patch train.py' cell below BEFORE launching training.")
else:
    print(f"  ✅ /dev/shm adequate for {shm_safe_workers} workers without fork patch")

# ── 3. Pick batch_size + gradient accumulation ───────────────────────────
if usable_mb >= 20000:
    batch_size, grad_accum = 768, 1
elif usable_mb >= 16000:
    batch_size, grad_accum = 512, 1
elif usable_mb >= 12000:
    batch_size, grad_accum = 384, 1
elif usable_mb >= 8000:
    batch_size, grad_accum = 256, 1
elif usable_mb >= 5000:
    batch_size, grad_accum = 128, 2
else:
    batch_size, grad_accum = 64, 4

effective_batch = batch_size * grad_accum

# ── 4. Choose num_workers ─────────────────────────────────────────────────
# After the fork-context patch, workers inherit parent memory and /dev/shm
# is never touched, so the CPU-based limit is the only relevant constraint.
# Before the patch, cap to shm_safe_workers as a safety net.
cpu_count = os.cpu_count() or 1
cpu_max_workers = min(4, max(0, cpu_count - 2))

# Detect whether the fork-context patch has already been applied.
_train_py_src = (PROJECT_ROOT / "scripts" / "train.py").read_text()
_fork_patch_applied = '"fork"' in _train_py_src and "cfg_nw" in _train_py_src

if _fork_patch_applied:
    num_workers = cpu_max_workers   # fork context → shm irrelevant
    print(f"\nFork-context patch detected → using full cpu_max_workers={num_workers}")
else:
    num_workers = min(cpu_max_workers, shm_safe_workers)
    print(f"\nFork-context patch NOT yet applied → capping workers to shm_safe={num_workers}")
    print("  Run the 'Patch train.py' cell then re-run Step 11 for full worker count.")

print()
print(f"╔══════════════════════════════════════════════╗")
print(f"║  batch_size       = {batch_size:<6}                   ║")
print(f"║  grad_accum       = {grad_accum:<6}                   ║")
print(f"║  effective_batch   = {effective_batch:<6}                  ║")
print(f"║  num_workers      = {num_workers:<6}   (written to YAML)   ║")
print(f"╚══════════════════════════════════════════════╝")
print()

# ── 5. Read base config and patch ────────────────────────────────────────
base_cfg_path = CONFIGS_DIR / "kaggle_config.yaml"
ws_cfg_path   = CONFIGS_DIR / "workstation_config.yaml"

if base_cfg_path.exists():
    with open(base_cfg_path) as f:
        cfg = yaml.safe_load(f)

    def patch_dict(d, patches):
        if not isinstance(d, dict):
            return
        for k, v in patches.items():
            if k in d:
                print(f"  patching {k}: {d[k]} → {v}")
                d[k] = v
        for child in d.values():
            if isinstance(child, dict):
                patch_dict(child, patches)

    patches = {
        "batch_size":                  batch_size,
        "num_workers":                 num_workers,   # train.py reads this after patch
        "gradient_accumulation_steps": grad_accum,
        "pin_memory":                  True,
        "persistent_workers":          num_workers > 0,
    }

    print("Patching config:")
    patch_dict(cfg, patches)

    if "gradient_accumulation_steps" not in str(cfg):
        target = cfg.get("training", cfg)
        target["gradient_accumulation_steps"] = grad_accum
        print(f"  added gradient_accumulation_steps: {grad_accum} (new key)")

    with open(ws_cfg_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

    print(f"\n✅ Wrote {ws_cfg_path}")
else:
    print(f"⚠️  {base_cfg_path} not found")

# ── 6. RTX 4090 performance flags ────────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True
print("✅ TF32 matmul ✓ · TF32 cuDNN ✓ · cuDNN benchmark ✓")

GPU VRAM: 24,564 MiB total · 110 MiB used · 23,963 MiB free
Usable for training: 21,915 MiB (after 2 GB safety margin)

/dev/shm  total=63 MB  free=63 MB
  ⚠️  /dev/shm is tight (63 MB free).  Without the fork-context
     patch, multi-worker DataLoading WILL crash (Bus Error / errno 28).
     Run the 'Patch train.py' cell below BEFORE launching training.

Fork-context patch detected → using full cpu_max_workers=4

╔══════════════════════════════════════════════╗
║  batch_size       = 768                      ║
║  grad_accum       = 1                        ║
║  effective_batch   = 768                     ║
║  num_workers      = 4        (written to YAML)   ║
╚══════════════════════════════════════════════╝

Patching config:
  patching batch_size: 256 → 768
  patching num_workers: 4 → 4
  patching pin_memory: True → True
  added gradient_accumulation_steps: 1 (new key)

✅ Wrote /home/mluser/projects_yash/new_project/PromptGFM-Bio/configs/workstation_config.yaml
✅ TF32 matmul ✓ · TF32 c

In [19]:
# #New in TE v6

# # ══════════════════════════════════════════════════════════════════════════

# # 11.5  Patch train.py  (run once per session; idempotent)

# #

# # Bug A — train.py hard-codes num_workers from cpu_count and ignores

# # config['data']['num_workers'], so YAML changes have no effect.

# # Bug B — Default multiprocessing context uses /dev/shm for tensor IPC.

# # With ~50 MB × 4 workers × batch prefetch the 64 MB default shm

# # exhausts instantly → Bus Error / errno ENOSPC (28).

# #

# # Fix A  — Make create_dataloaders() respect config['data']['num_workers']

# # when the key is present.

# # Fix B  — Pass multiprocessing_context='fork' to both DataLoaders.

# # Fork workers *inherit* the parent's address space; the large

# # cached_node_features and cached_edge_index tensors are already

# # in memory and never go through /dev/shm.

# # ══════════════════════════════════════════════════════════════════════════

# import warnings\
# from pathlib import Path

# train_py = PROJECT_ROOT / "scripts" / "train.py"\
# src_orig = src = train_py.read_text()

# # ── Patch A: respect config num_workers ──────────────────────────────────

# _OLD_A = '''\
# if is_windows:\
# num_workers = 0  # Windows multiprocessing can't pickle nested collate_fn\
# logger.info(f"  DataLoader workers: {num_workers} (Windows: single-threaded to avoid pickling issues)")\
# else:\
# num_workers = min(4, max(0, cpu_count - 2)) if cpu_count and cpu_count > 2 else 0\
# logger.info(f"  DataLoader workers: {num_workers} (parallel data loading enabled)")'''

# _NEW_A = '''\
# # Read num_workers from config when explicitly set; otherwise auto-detect.\
# _cfg_nw = config.get("data", {}).get("num_workers", None)\
# cfg_nw = int(_cfg_nw) if _cfg_nw is not None else None\
# if is_windows:\
# num_workers = 0\
# logger.info(f"  DataLoader workers: {num_workers} (Windows)")\
# elif cfg_nw is not None:\
# num_workers = cfg_nw\
# logger.info(f"  DataLoader workers: {num_workers} (from config)")\
# else:\
# num_workers = min(4, max(0, cpu_count - 2)) if cpu_count and cpu_count > 2 else 0\
# logger.info(f"  DataLoader workers: {num_workers} (auto-detected)")'''

# if _OLD_A in src:\
# src = src.replace(_OLD_A, _NEW_A)\
# print("✅ Patch A applied — train.py now honours config['data']['num_workers']")\
# elif "cfg_nw" in src:\
# print("ℹ️  Patch A already applied")\
# else:\
# print("⚠️  Patch A: target string not found — train.py may have changed; skipping")\
# '''

# # ── Patch B: fork context for both DataLoaders ───────────────────────────

# # We target the closing lines that are identical in both train_loader and

# # val_loader definitions.

# _OLD_B = '''\
# pin_memory=True if torch.cuda.is_available() else False,  # Faster GPU transfer\
# persistent_workers=True if num_workers > 0 else False  # Keep workers alive\
# )'''

# _NEW_B = '''\
# pin_memory=True if torch.cuda.is_available() else False,\
# persistent_workers=True if num_workers > 0 else False,\
# # 'fork' workers inherit parent memory; large cached tensors (node_features,\
# # edge_index) are never serialised to /dev/shm — eliminates Bus Error / ENOSPC.\
# multiprocessing_context="fork" if num_workers > 0 else None,\
# )'''

# _count_b = src.count(_OLD_B)\
# if _count_b == 2:\
# src = src.replace(_OLD_B, _NEW_B)\
# print("✅ Patch B applied — both DataLoaders use fork context (bypasses /dev/shm)")\
# elif _count_b == 0 and '"fork"' in src:\
# print("ℹ️  Patch B already applied")\
# elif _count_b == 0:\
# print("⚠️  Patch B: target string not found — train.py may have changed; skipping")\
# else:\
# print(f"⚠️  Patch B: found {_count_b} matches (expected 2) — skipping to be safe")\
# '''

# # ── Patch B: fork context for both DataLoaders ───────────────────────────

# _OLD_B_VARIANTS = [\
# '''\
# pin_memory=True if torch.cuda.is_available() else False,  # Faster GPU transfer\
# persistent_workers=True if num_workers > 0 else False  # Keep workers alive\
# )''',

# '''
#     pin_memory=True if torch.cuda.is_available() else False,
#     persistent_workers=True if num_workers > 0 else False
# )'''
# ]

# _NEW_B = '''\
# pin_memory=True if torch.cuda.is_available() else False,\
# persistent_workers=True if num_workers > 0 else False,\
# multiprocessing_context="fork" if num_workers > 0 else None,\
# )'''

# patched = 0\
# for old in _OLD_B_VARIANTS:\
# if old in src:\
# src = src.replace(old, _NEW_B)\
# patched += 1

# if patched == 2:\
# print("✅ Patch B applied — both DataLoaders patched")\
# elif patched == 1:\
# print("⚠️  Patch B partially applied (1 block patched)")\
# elif '"fork"' in src:\
# print("ℹ️  Patch B already applied")\
# else:\
# print("⚠️  Patch B: no match found — train.py structure changed")

# # ── Suppress PyG ABI UserWarnings that flood stdout ──────────────────────

# # Insert a warnings filter immediately after the torch_geometric import block.

# _OLD_C = "from src.data.dataset import BiomedicalGraphDataset, GeneDiseaseDataset"\
# _NEW_C = """\
# import warnings as _w\
# _w.filterwarnings("ignore", message="An issue occurred while importing", category=UserWarning)

# from src.data.dataset import BiomedicalGraphDataset, GeneDiseaseDataset"""

# if _OLD_C in src and "_w.filterwarnings" not in src:\
# src = src.replace(_OLD_C, _NEW_C)\
# print("✅ Patch C applied — PyG ABI UserWarnings suppressed in train.py output")\
# elif "_w.filterwarnings" in src:\
# print("ℹ️  Patch C already applied")\
# else:\
# print("⚠️  Patch C: anchor not found — skipping")

# # ── Write & verify ────────────────────────────────────────────────────────

# if src != src_orig:\
# train_py.write_text(src)\
# print(f"\n✅ train.py saved ({train_py})")\
# print("   Re-run Step 11 (VRAM Config) to update num_workers in the YAML now")\
# print("   that the fork-patch is detected.")\
# else:\
# print("\nℹ️  No changes written — all patches already present")

# # Quick smoke-test: confirm multiprocessing context setting is present

# assert '"fork"' in train_py.read_text(), "Patch B verification failed"\
# print("✅ Patch verification passed")

In [20]:
# #New in TE v6 (CLEAN FIX)
# # ══════════════════════════════════════════════════════════════════════════
# # 11.5  Patch train.py (robust + no syntax issues)
# # ══════════════════════════════════════════════════════════════════════════

# from pathlib import Path

# train_py = PROJECT_ROOT / "scripts" / "train.py"
# src_orig = src = train_py.read_text()

# # ── Patch A: respect config num_workers ──────────────────────────────────
# OLD_A = """\
#     if is_windows:
#         num_workers = 0  # Windows multiprocessing can't pickle nested collate_fn
#         logger.info(f"  DataLoader workers: {num_workers} (Windows: single-threaded to avoid pickling issues)")
#     else:
#         num_workers = min(4, max(0, cpu_count - 2)) if cpu_count and cpu_count > 2 else 0
#         logger.info(f"  DataLoader workers: {num_workers} (parallel data loading enabled)")"""

# NEW_A = """\
#     # Read num_workers from config when explicitly set; otherwise auto-detect.
#     _cfg_nw = config.get("data", {}).get("num_workers", None)
#     cfg_nw = int(_cfg_nw) if _cfg_nw is not None else None
#     if is_windows:
#         num_workers = 0
#         logger.info(f"  DataLoader workers: {num_workers} (Windows)")
#     elif cfg_nw is not None:
#         num_workers = cfg_nw
#         logger.info(f"  DataLoader workers: {num_workers} (from config)")
#     else:
#         num_workers = min(4, max(0, cpu_count - 2)) if cpu_count and cpu_count > 2 else 0
#         logger.info(f"  DataLoader workers: {num_workers} (auto-detected)")"""

# if OLD_A in src:
#     src = src.replace(OLD_A, NEW_A)
#     print("✅ Patch A applied")
# elif "cfg_nw" in src:
#     print("ℹ️ Patch A already applied")
# else:
#     print("⚠️ Patch A not found")

# # ── Patch B: fork context (handles BOTH variants) ────────────────────────
# OLD_B_VARIANTS = [
#     """\
#         pin_memory=True if torch.cuda.is_available() else False,  # Faster GPU transfer
#         persistent_workers=True if num_workers > 0 else False  # Keep workers alive
#     )""",
#     """\
#         pin_memory=True if torch.cuda.is_available() else False,
#         persistent_workers=True if num_workers > 0 else False
#     )"""
# ]

# NEW_B = """\
#         pin_memory=True if torch.cuda.is_available() else False,
#         persistent_workers=True if num_workers > 0 else False,
#         multiprocessing_context="fork" if num_workers > 0 else None,
#     )"""

# patched = 0
# for old in OLD_B_VARIANTS:
#     if old in src:
#         src = src.replace(old, NEW_B)
#         patched += 1

# if patched == 2:
#     print("✅ Patch B applied (both loaders)")
# elif patched == 1:
#     print("⚠️ Patch B applied partially (1 loader)")
# elif 'multiprocessing_context="fork"' in src:
#     print("ℹ️ Patch B already applied")
# else:
#     print("⚠️ Patch B not applied")

# # ── Patch C: suppress PyG warnings ───────────────────────────────────────
# OLD_C = "from src.data.dataset import BiomedicalGraphDataset, GeneDiseaseDataset"
# NEW_C = """\
# import warnings as _w
# _w.filterwarnings("ignore", message="An issue occurred while importing", category=UserWarning)

# from src.data.dataset import BiomedicalGraphDataset, GeneDiseaseDataset"""

# if OLD_C in src and "_w.filterwarnings" not in src:
#     src = src.replace(OLD_C, NEW_C)
#     print("✅ Patch C applied")
# elif "_w.filterwarnings" in src:
#     print("ℹ️ Patch C already applied")
# else:
#     print("⚠️ Patch C not found")

# # ── Write file ───────────────────────────────────────────────────────────
# if src != src_orig:
#     train_py.write_text(src)
#     print(f"\n✅ train.py updated")
# else:
#     print("\nℹ️ No changes needed")

# # ── Verification (robust) ────────────────────────────────────────────────
# content = train_py.read_text()

# if 'multiprocessing_context="fork"' not in content:
#     raise RuntimeError("❌ Fork patch missing")

# if "cfg_nw" not in content:
#     raise RuntimeError("❌ num_workers patch missing")

# print("✅ Patch verification passed")

In [21]:
# #New in TE v6 (FIXED)
# # ══════════════════════════════════════════════════════════════════════════
# # 11.5  Patch train.py  (robust + compatible with your file)
# # ══════════════════════════════════════════════════════════════════════════

# from pathlib import Path

# train_py = PROJECT_ROOT / "scripts" / "train.py"
# src_orig = src = train_py.read_text()

# # ── Patch A: respect config num_workers ──────────────────────────────────
# _OLD_A = '''\
#     if is_windows:
#         num_workers = 0  # Windows multiprocessing can't pickle nested collate_fn
#         logger.info(f"  DataLoader workers: {num_workers} (Windows: single-threaded to avoid pickling issues)")
#     else:
#         num_workers = min(4, max(0, cpu_count - 2)) if cpu_count and cpu_count > 2 else 0
#         logger.info(f"  DataLoader workers: {num_workers} (parallel data loading enabled)")'''

# _NEW_A = '''\
#     # Read num_workers from config when explicitly set; otherwise auto-detect.
#     _cfg_nw = config.get("data", {}).get("num_workers", None)
#     cfg_nw = int(_cfg_nw) if _cfg_nw is not None else None
#     if is_windows:
#         num_workers = 0
#         logger.info(f"  DataLoader workers: {num_workers} (Windows)")
#     elif cfg_nw is not None:
#         num_workers = cfg_nw
#         logger.info(f"  DataLoader workers: {num_workers} (from config)")
#     else:
#         num_workers = min(4, max(0, cpu_count - 2)) if cpu_count and cpu_count > 2 else 0
#         logger.info(f"  DataLoader workers: {num_workers} (auto-detected)")'''

# if _OLD_A in src:
#     src = src.replace(_OLD_A, _NEW_A)
#     print("✅ Patch A applied — train.py now honours config['data']['num_workers']")
# elif "cfg_nw" in src:
#     print("ℹ️  Patch A already applied")
# else:
#     print("⚠️  Patch A: target string not found — skipping")

# # ── Patch B: fork context (FIXED for your train.py) ──────────────────────
# _OLD_B_VARIANTS = [
#     '''\
#         pin_memory=True if torch.cuda.is_available() else False,  # Faster GPU transfer
#         persistent_workers=True if num_workers > 0 else False  # Keep workers alive
#     )''',

#     '''\
#         pin_memory=True if torch.cuda.is_available() else False,
#         persistent_workers=True if num_workers > 0 else False
#     )'''
# ]

# _NEW_B = '''\
#         pin_memory=True if torch.cuda.is_available() else False,
#         persistent_workers=True if num_workers > 0 else False,
#         # Fork avoids /dev/shm → fixes Bus Error / ENOSPC
#         multiprocessing_context="fork" if num_workers > 0 else None,
#     )'''

# patched = 0
# for old in _OLD_B_VARIANTS:
#     if old in src:
#         src = src.replace(old, _NEW_B)
#         patched += 1

# if patched == 2:
#     print("✅ Patch B applied — both DataLoaders patched")
# elif patched == 1:
#     print("⚠️  Patch B partially applied (1 block patched)")
# elif '"fork"' in src:
#     print("ℹ️  Patch B already applied")
# else:
#     print("⚠️  Patch B: no match found — skipping")

# # ── Patch C: suppress PyG warnings ───────────────────────────────────────
# _OLD_C = "from src.data.dataset import BiomedicalGraphDataset, GeneDiseaseDataset"
# _NEW_C = """\
# import warnings as _w
# _w.filterwarnings("ignore", message="An issue occurred while importing", category=UserWarning)

# from src.data.dataset import BiomedicalGraphDataset, GeneDiseaseDataset"""

# if _OLD_C in src and "_w.filterwarnings" not in src:
#     src = src.replace(_OLD_C, _NEW_C)
#     print("✅ Patch C applied — warnings suppressed")
# elif "_w.filterwarnings" in src:
#     print("ℹ️  Patch C already applied")
# else:
#     print("⚠️  Patch C: anchor not found — skipping")

# # ── Write file ───────────────────────────────────────────────────────────
# if src != src_orig:
#     train_py.write_text(src)
#     print(f"\n✅ train.py saved ({train_py})")
# else:
#     print("\nℹ️  No changes written — already patched")

# # ── Robust verification (FIXED) ──────────────────────────────────────────
# content = train_py.read_text()

# if 'multiprocessing_context="fork"' not in content:
#     raise RuntimeError("❌ Patch B verification failed — fork context not found")

# if "cfg_nw" not in content:
#     raise RuntimeError("❌ Patch A verification failed")

# print("✅ Patch verification passed")

In [22]:
# ══════════════════════════════════════════════════════════════════════════
# 11.6  Root-cause fix: remove graph tensors from DataLoader batch payload
#
# DIAGNOSIS
# ---------
# Even with fork context, PyTorch uses fd-based shared memory when workers
# send data back to the main process through the multiprocessing queue.
# The collate function currently returns two graph-wide constant tensors
# in EVERY batch:
#
#   node_features : shape [19576, 128] float32  →  ~10 MB per batch
#   edge_index    : shape [2, 1854012] int64    →  ~30 MB per batch
#
# With num_workers=4, prefetch_factor=2 that is:
#   8 in-flight batches × 40 MB = 320 MB through /dev/shm  → ENOSPC crash
#
# These tensors are IDENTICAL in every batch (they are the full graph).
# They should live on the trainer, not inside the DataLoader queue.
#
# PATCHES
# -------
# Patch D — train.py collate_fn:
#   Stop returning node_features / edge_index in the batch dict.
#
# Patch E — train.py run_finetuning:
#   After constructing the trainer, call trainer.set_graph_tensors()
#   to store the graph on the correct device once.
#
# Patch F — src/training/finetune.py:
#   • Add set_graph_tensors() method.
#   • In train_epoch: get node_features/edge_index from self when absent
#     from batch (backward-compatible: still works if present in batch).
# ══════════════════════════════════════════════════════════════════════════
from pathlib import Path

train_py    = PROJECT_ROOT / "scripts" / "train.py"
finetune_py = PROJECT_ROOT / "src" / "training" / "finetune.py"

# ─────────────────────────────────────────────────────────────────────────
# Patch D — remove large constants from collate return dict
# ─────────────────────────────────────────────────────────────────────────
src_train = train_py.read_text()

_OLD_D = """        return {
            'node_features': node_features,
            'edge_index': edge_index,
            'disease_texts': combined_disease_texts,
            'gene_indices': gene_indices,
            'labels': labels
        }"""

_NEW_D = """        # node_features and edge_index are deliberately NOT included here.
        # They are graph-wide constants (~40 MB combined) that never change
        # between batches.  Returning them through the DataLoader queue forces
        # workers to serialize them into /dev/shm on every prefetch, which
        # exhausts shared memory and causes ENOSPC Bus Error crashes.
        # The trainer receives them once via set_graph_tensors() below.
        return {
            'disease_texts': combined_disease_texts,
            'gene_indices': gene_indices,
            'labels': labels
        }"""

if _OLD_D in src_train:
    src_train = src_train.replace(_OLD_D, _NEW_D)
    print("✅ Patch D applied — node_features/edge_index removed from batch dict")
elif "'node_features': node_features" not in src_train:
    print("ℹ️  Patch D already applied")
else:
    print("⚠️  Patch D: exact string not found — check indentation in collate_fn")

# ─────────────────────────────────────────────────────────────────────────
# Patch E — call set_graph_tensors() after trainer construction
# Insert after the existing  `if emb_cache: trainer.set_prompt_cache(...)` block
# ─────────────────────────────────────────────────────────────────────────
_OLD_E = """    if emb_cache:
        trainer.set_prompt_cache(emb_cache)"""

_NEW_E = """    if emb_cache:
        trainer.set_prompt_cache(emb_cache)

    # ── Graph tensors: pass once to trainer, never through DataLoader queue ──
    # Resolve node features from the graph (same logic as create_collate_fn).
    _graph = dataset.graph
    _num_g = _graph['gene'].num_nodes
    if hasattr(_graph['gene'], 'x') and _graph['gene'].x is not None:
        _nf = _graph['gene'].x
    else:
        import torch as _t
        _t.manual_seed(42)
        _nf = _t.randn(_num_g, config.get('model', {}).get('gene_feature_dim', 128))

    # Resolve edge index (gene-gene PPI edges).
    _et = _graph.edge_types if hasattr(_graph, 'edge_types') else []
    _ei = None
    for _candidate in [('gene','interacts','gene'),
                        ('gene','protein_interaction','gene'),
                        ('gene','ppi','gene')]:
        if _candidate in _et:
            _ei = _graph[_candidate].edge_index
            break
    if _ei is None:
        import torch as _t
        _ei = _t.empty((2, 0), dtype=_t.long)

    trainer.set_graph_tensors(_nf, _ei)
    logger.info(f"  ✅ Graph tensors set on trainer "
                f"(node_features={list(_nf.shape)}, "
                f"edge_index={list(_ei.shape)}) — removed from DataLoader queue")"""

if _OLD_E in src_train:
    src_train = src_train.replace(_OLD_E, _NEW_E)
    print("✅ Patch E applied — set_graph_tensors() called after trainer construction")
elif "set_graph_tensors" in src_train and "Graph tensors: pass once" in src_train:
    print("ℹ️  Patch E already applied")
else:
    print("⚠️  Patch E: anchor not found — trainer.set_prompt_cache() line may differ")

train_py.write_text(src_train)
print(f"   train.py saved")

# ─────────────────────────────────────────────────────────────────────────
# Patch F — finetune.py: add set_graph_tensors() + use in train_epoch
# ─────────────────────────────────────────────────────────────────────────
src_ft = finetune_py.read_text()

# F1 — add set_graph_tensors method alongside set_prompt_cache
_OLD_F1 = """    def set_prompt_cache(self, cache: dict):"""

_NEW_F1 = """    def set_graph_tensors(self, node_features, edge_index):
        \"\"\"Store graph-wide tensors on the trainer so the DataLoader never
        needs to carry them inside batch dicts.  Called once after construction.\"\"\"
        self.graph_node_features = node_features.to(self.device)
        self.graph_edge_index    = edge_index.to(self.device)
        logger.info(f"  Graph tensors moved to {self.device}: "
                    f"node_features={list(node_features.shape)}, "
                    f"edge_index={list(edge_index.shape)}")

    def set_prompt_cache(self, cache: dict):"""

if _OLD_F1 in src_ft:
    src_ft = src_ft.replace(_OLD_F1, _NEW_F1)
    print("✅ Patch F1 applied — set_graph_tensors() method added to PromptGFMTrainer")
elif "set_graph_tensors" in src_ft:
    print("ℹ️  Patch F1 already applied")
else:
    # Fallback: insert before set_prompt_cache regardless of exact signature
    _fallback_anchor = "def set_prompt_cache"
    if _fallback_anchor in src_ft:
        src_ft = src_ft.replace(
            _fallback_anchor,
            "def set_graph_tensors(self, node_features, edge_index):\n"
            "        self.graph_node_features = node_features.to(self.device)\n"
            "        self.graph_edge_index    = edge_index.to(self.device)\n\n"
            "    " + _fallback_anchor
        )
        print("✅ Patch F1 applied (fallback path)")
    else:
        print("⚠️  Patch F1: could not locate set_prompt_cache — add set_graph_tensors manually")

# F2 — in train_epoch, get node_features/edge_index from self when not in batch
# Locate the lines that extract these two keys from the batch dict.
_OLD_F2 = """            node_features = batch['node_features'].to(self.device)
            edge_index = batch['edge_index'].to(self.device)"""

_NEW_F2 = """            # Graph-wide constants live on the trainer (set_graph_tensors),
            # not in the batch dict — keeps DataLoader queue lean.
            node_features = (batch['node_features'].to(self.device)
                             if 'node_features' in batch
                             else self.graph_node_features)
            edge_index    = (batch['edge_index'].to(self.device)
                             if 'edge_index' in batch
                             else self.graph_edge_index)"""

if _OLD_F2 in src_ft:
    src_ft = src_ft.replace(_OLD_F2, _NEW_F2)
    print("✅ Patch F2 applied — train_epoch uses trainer-cached graph tensors")
elif "self.graph_node_features" in src_ft:
    print("ℹ️  Patch F2 already applied")
else:
    # Try alternate indentation (4-space vs 8-space outer function body)
    _OLD_F2b = """        node_features = batch['node_features'].to(self.device)
        edge_index = batch['edge_index'].to(self.device)"""
    _NEW_F2b = """        node_features = (batch['node_features'].to(self.device)
                         if 'node_features' in batch
                         else self.graph_node_features)
        edge_index    = (batch['edge_index'].to(self.device)
                         if 'edge_index' in batch
                         else self.graph_edge_index)"""
    if _OLD_F2b in src_ft:
        src_ft = src_ft.replace(_OLD_F2b, _NEW_F2b)
        print("✅ Patch F2 applied (alternate indentation)")
    else:
        print("⚠️  Patch F2: could not find node_features/edge_index extraction in train_epoch")
        print("   Open finetune.py and manually replace:")
        print("     batch['node_features'].to(self.device)")
        print("   with:")
        print("     batch.get('node_features', self.graph_node_features).to(self.device)  # etc.")

# Also apply the same backward-compatible pattern for validate_epoch if present
if "'node_features'" in src_ft and "validate_epoch" in src_ft:
    # Only patch lines inside validate_epoch that still do dict extraction
    _OLD_F3 = """            node_features = batch['node_features'].to(self.device)
            edge_index = batch['edge_index'].to(self.device)"""
    # This is the same as _OLD_F2 but may appear a second time in validate_epoch
    count_old = src_ft.count(_OLD_F3)
    if count_old > 0:
        src_ft = src_ft.replace(_OLD_F3, _NEW_F2)   # replace ALL remaining occurrences
        print(f"✅ Patch F3 applied — validate_epoch also updated ({count_old} site(s))")

finetune_py.write_text(src_ft)
print(f"   finetune.py saved")

# ─────────────────────────────────────────────────────────────────────────
# Verification
# ─────────────────────────────────────────────────────────────────────────
print()
print("─── Verification ───")
_t_src = train_py.read_text()
_f_src = finetune_py.read_text()

checks = {
    "batch dict no longer has node_features key":
        "'node_features': node_features" not in _t_src,
    "batch dict no longer has edge_index key":
        "'edge_index': edge_index" not in _t_src,
    "set_graph_tensors() called in run_finetuning":
        "trainer.set_graph_tensors(" in _t_src,
    "set_graph_tensors() defined in finetune.py":
        "def set_graph_tensors(" in _f_src,
    "train_epoch uses self.graph_node_features":
        "self.graph_node_features" in _f_src,
}

all_ok = True
for desc, result in checks.items():
    status = "✅" if result else "❌"
    if not result:
        all_ok = False
    print(f"  {status}  {desc}")

print()
if all_ok:
    print("✅ All patches verified — proceed to training")
    print()
    print("Expected outcome:")
    print("  • DataLoader queue payload per batch: ~4 KB (indices + labels only)")
    print("  • /dev/shm usage: ~0 bytes from DataLoader")
    print("  • Graph tensors moved to GPU once at trainer init, reused every batch")
else:
    print("⚠️  Some patches need manual review — check the lines marked ❌ above")

ℹ️  Patch D already applied
✅ Patch E applied — set_graph_tensors() called after trainer construction
   train.py saved
ℹ️  Patch F1 already applied
ℹ️  Patch F2 already applied
   finetune.py saved

─── Verification ───
  ✅  batch dict no longer has node_features key
  ✅  batch dict no longer has edge_index key
  ✅  set_graph_tensors() called in run_finetuning
  ✅  set_graph_tensors() defined in finetune.py
  ✅  train_epoch uses self.graph_node_features

✅ All patches verified — proceed to training

Expected outcome:
  • DataLoader queue payload per batch: ~4 KB (indices + labels only)
  • /dev/shm usage: ~0 bytes from DataLoader
  • Graph tensors moved to GPU once at trainer init, reused every batch


## 12. Train

Uses `configs/workstation_config.yaml` auto-generated above with VRAM-aware batch_size.

In [ ]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"
print("✅ Offline mode confirmed")

In [ ]:
import torch, subprocess, sys

# ── Config ────────────────────────────────────────────────────────────────
ws_cfg = CONFIGS_DIR / "workstation_config.yaml"
config = str(ws_cfg) if ws_cfg.exists() else str(CONFIGS_DIR / "kaggle_config.yaml")
print(f"Using config: {config}")

# ── Auto-detect checkpoints for resume ────────────────────────────────────
ckpts = sorted(
    CKPT_DIR.glob("checkpoint_epoch_*.pt"),
    key=lambda p: int(p.stem.split("_")[-1])
) if CKPT_DIR.exists() else []

RESUME = RESUME_CHECKPOINTS or bool(ckpts)

base_args = [str(SCRIPTS_DIR / "train.py"), "--config", config]
if RESUME and ckpts:
    last_ckpt = str(ckpts[-1])
    base_args += ["--resume-checkpoint", last_ckpt]
    print(f"Resuming from: {last_ckpt}")
elif RESUME:
    print("RESUME_CHECKPOINTS=True but no checkpoints found — starting fresh")
    RESUME = False

# ── Launch ────────────────────────────────────────────────────────────────
n_gpus = torch.cuda.device_count()
print(f"GPUs available: {n_gpus}")

if n_gpus > 1:
    import shutil
    torchrun = shutil.which("torchrun") or "torchrun"
    cmd = [torchrun, f"--nproc_per_node={n_gpus}", "--master_port=29500"] + base_args
    print(f"Launching DDP on {n_gpus} GPUs")
else:
    cmd = [sys.executable] + base_args
    print("Single-GPU launch (RTX 4090)")

print("Running:", " ".join(cmd))
print()
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
print("\nTraining exit code:", result.returncode)

## 12.5. 📊 Post-Training GPU Check
See peak memory usage — helps decide if batch_size can go higher next time.

In [ ]:
import subprocess, torch

subprocess.run(["nvidia-smi"])

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated(0) / 1e9
    reserved = torch.cuda.max_memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print()
    print(f"PyTorch peak allocated : {peak:.1f} GB")
    print(f"PyTorch peak reserved  : {reserved:.1f} GB")
    print(f"Total VRAM             : {total:.1f} GB")
    print(f"Headroom               : {total - reserved:.1f} GB")
    if total - reserved > 4.0:
        print("💡 You have headroom — try a larger batch_size next run.")
    elif total - reserved < 1.0:
        print("⚠️  Very tight — reduce batch_size if you see OOM errors.")

## 13. 💾 Backup to GitHub / HuggingFace

> **CRITICAL**: Workstation wipes all data after 5 days!

In [ ]:
import subprocess, os

os.environ["GIT_TERMINAL_PROMPT"] = "0"

if not (PROJECT_ROOT / ".git").is_dir():
    print("⚠️  Not a git repo — cannot push. Back up manually:")
    print(f"   scp -r {CKPT_DIR} your-laptop:~/backups/")
else:
    for pattern in ["checkpoints/", "logs/"]:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "add", "-f", pattern])

    status = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"],
        capture_output=True, text=True
    )

    if status.stdout.strip():
        ckpts = sorted(
            CKPT_DIR.glob("checkpoint_epoch_*.pt"),
            key=lambda f: int(f.stem.split("_")[-1])
        ) if CKPT_DIR.exists() else []
        epoch = ckpts[-1].stem.split("_")[-1] if ckpts else "?"

        subprocess.run([
            "git", "-C", str(PROJECT_ROOT), "commit", "-m",
            f"Workstation training: epoch {epoch} checkpoints"
        ])
        result = subprocess.run(["git", "-C", str(PROJECT_ROOT), "push"])
        if result.returncode == 0:
            print("✅ Pushed to GitHub")
        else:
            print("❌ Git push failed — check GITHUB_TOKEN in .env")
    else:
        print("Nothing new to commit")

print()
print("For large files, consider HuggingFace Hub:")
print(f"  huggingface-cli upload your-username/promptgfm-bio {HF_CACHE_DIR} --repo-type model")

## 14. Quick Evaluation

In [ ]:
'''
import subprocess, sys

ws_cfg = CONFIGS_DIR / "workstation_config.yaml"
config = str(ws_cfg) if ws_cfg.exists() else str(CONFIGS_DIR / "kaggle_config.yaml")

best = CKPT_DIR / "best_model.pt"
if not best.exists():
    print("No best_model.pt yet — run more training epochs first")
else:
    result = subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / "evaluate.py"),
         "--config", config,
         "--checkpoint", str(best)],
        cwd=str(PROJECT_ROOT),
    )
    print("Evaluation exit code:", result.returncode)
'''

In [ ]:
import subprocess, sys

# 🔴 FORCE correct config (no auto-switching)
#config = str(CONFIGS_DIR / "workstation_config.yaml")

best = CKPT_DIR / "best_model.pt"

if not best.exists():
    print("No best_model.pt yet — run more training epochs first")
else:
    result = subprocess.run(
        [
            sys.executable,
            str(SCRIPTS_DIR / "evaluate.py"),
            "--config", config,
            "--checkpoint", str(best),
        ],
        cwd=str(PROJECT_ROOT),
    )
    print("Evaluation exit code:", result.returncode)

## 15. Disk Usage Check

In [ ]:
import subprocess
from pathlib import Path

subprocess.run(["df", "-h", str(PROJECT_ROOT)])
print()

for label, path in [("hf_cache", HF_CACHE_DIR), ("data", DATA_DIR),
                     ("checkpoints", CKPT_DIR), ("logs", LOGS_DIR)]:
    p = Path(path)
    if p.exists():
        result = subprocess.run(["du", "-sh", str(p)], capture_output=True, text=True)
        print(result.stdout.strip())

## 16. 🔧 Manual Override (Optional)

If auto-detected batch_size isn't right, uncomment and edit below, then re-run Step 12.

In [ ]:
# import yaml
#
# MANUAL_BATCH_SIZE  = 384
# MANUAL_GRAD_ACCUM  = 1
# MANUAL_NUM_WORKERS = 4
#
# ws_cfg_path = PROJECT_ROOT / "configs" / "workstation_config.yaml"
# if ws_cfg_path.exists():
#     with open(ws_cfg_path) as f:
#         cfg = yaml.safe_load(f)
#
#     def patch(d, k, v):
#         if isinstance(d, dict):
#             if k in d: d[k] = v
#             for child in d.values(): patch(child, k, v)
#
#     patch(cfg, "batch_size", MANUAL_BATCH_SIZE)
#     patch(cfg, "gradient_accumulation_steps", MANUAL_GRAD_ACCUM)
#     patch(cfg, "num_workers", MANUAL_NUM_WORKERS)
#
#     with open(ws_cfg_path, "w") as f:
#         yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
#     print(f"✅ Updated: batch={MANUAL_BATCH_SIZE}, accum={MANUAL_GRAD_ACCUM}")